# Speak Like Me — preference candidate generation

This notebook is independent of the SFT training notebook. It loads the finished private LoRA adapter and generates candidates for 100 held-out prompts from both the original base model and the SFT model. It also carries forward the historical self-authored reply as a blinded reference candidate. No API key is required for this stage.

## 1. Confirm the GPU runtime
Select **Runtime → Change runtime type → GPU** before running this cell.

In [ ]:
!nvidia-smi

## 2. Upload the private generation bundle
Upload only `preference-generation-colab.zip`. The archive contains the adapter, held-out prompts, rubric, and generation script.

In [ ]:
from google.colab import files
uploaded = files.upload()
assert 'preference-generation-colab.zip' in uploaded
!unzip -q preference-generation-colab.zip -d /content/speak-like-me
%cd /content/speak-like-me

## 3. Install the pinned training dependencies

In [ ]:
%pip install -q -r training/requirements.txt

## 4. Generate base and SFT candidates
The public base model will download in this fresh runtime. Generation then runs in two passes and prints progress after every batch.

In [ ]:
!python training/generate_preference_candidates.py

## 5. Validate the generated file

In [ ]:
import json
from pathlib import Path
candidate_path = Path('artifacts/preference/candidates.partial.jsonl')
rows = [json.loads(line) for line in candidate_path.read_text().splitlines() if line]
assert len(rows) == 100
assert all(len(row['candidates']) == 3 for row in rows)
assert all(candidate['text'].strip() for row in rows for candidate in row['candidates'])
print({'prompts': len(rows), 'candidates_per_prompt': 3, 'validation': 'passed'})

## 6. Download the partial candidate set
Keep this file private. It contains sanitized conversation context and generated replies.

In [ ]:
!zip -q -j preference-candidates-partial.zip artifacts/preference/candidates.partial.jsonl
files.download('preference-candidates-partial.zip')